# Model Comparison: GARCH vs LSTM vs Transformer

Loads saved predictions from all three models, aligns them on the 2024 test set, and produces a single metrics table and overlay plot.

**Files used:**
- `data/garch_results.csv` — GARCH rolling walk-forward forecasts (365 rows)
- `models/mse_lstm_model.pth` — LSTM weights trained with MSE loss
- `models/transformer_model.pth` — Transformer weights

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import sys
import os

DATA_PATH  = '../data/btc_data.csv'
GARCH_PATH = '../data/garch_results.csv'
LSTM_PATH  = '../models/mse_lstm_model.pth'
TRANS_PATH = '../models/transformer_model.pth'
WINDOW     = 30

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models.lstm import LSTMModel
from models.transformer import TransformerModel

## Data Pipeline

Replicates the preprocessing from `03_lstm.ipynb` and `04_transformer.ipynb`:
1. Load `btc_data.csv`
2. Fit `StandardScaler` on train (2018–2022) only
3. Apply scaling to test features; keep raw volatility as target
4. Build 30-day sliding windows → test sequences start at 2024-01-31

In [ ]:
btc_df    = pd.read_csv(DATA_PATH, index_col='Date', parse_dates=True)
train_raw = btc_df.loc['2018-01-31':'2022-12-31']
test_raw  = btc_df.loc['2024-01-01':'2024-12-31']

scaler = StandardScaler()
scaler.fit(train_raw[['log_returns', 'volatility']])

test_scaled = test_raw.copy()
test_scaled[['log_returns', 'volatility']] = scaler.transform(test_raw[['log_returns', 'volatility']])
test_scaled['volatility_target'] = test_raw['volatility']

X_data = test_scaled[['log_returns', 'volatility']].values
y_data = test_scaled['volatility_target'].values

X = np.array([X_data[i:i+WINDOW] for i in range(len(test_scaled) - WINDOW)])
y = np.array([y_data[i+WINDOW]   for i in range(len(test_scaled) - WINDOW)])

forecast_dates = test_raw.index[WINDOW:]
actual         = test_raw['volatility'].iloc[WINDOW:].values

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)
loader   = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=128, shuffle=False)

print(f"Test sequences: {len(X)} | {forecast_dates[0].date()} → {forecast_dates[-1].date()}")

## Load Predictions

In [ ]:
# GARCH: walk-forward forecasts saved from 02_garch.ipynb
garch_df = pd.read_csv(GARCH_PATH, index_col='Date', parse_dates=True)
print(f"GARCH: {len(garch_df)} rows | {garch_df.index[0].date()} → {garch_df.index[-1].date()}")

In [ ]:
# LSTM: load saved weights and run inference on test sequences
lstm_model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
lstm_model.load_state_dict(torch.load(LSTM_PATH, map_location='cpu'))
lstm_model.eval()

lstm_preds = []
with torch.no_grad():
    for X_batch, _ in loader:
        lstm_preds.append(lstm_model(X_batch))
lstm_preds = torch.cat(lstm_preds).numpy().flatten()

print(f"LSTM predictions: {len(lstm_preds)}")

In [ ]:
# Transformer: load saved weights and run inference on test sequences
trans_model = TransformerModel(d_model=64, seq_len=30, nhead=4, num_layers=2)
trans_model.load_state_dict(torch.load(TRANS_PATH, map_location='cpu'))
trans_model.eval()

trans_preds = []
with torch.no_grad():
    for X_batch, _ in loader:
        trans_preds.append(trans_model(X_batch))
trans_preds = torch.cat(trans_preds).numpy().flatten()

print(f"Transformer predictions: {len(trans_preds)}")

In [ ]:
# LSTM/Transformer: 335 rows (2024-01-31 onward)
# GARCH: 365 rows (full 2024). Inner join → 335 aligned rows.
dl_df = pd.DataFrame({
    'actual':      actual,
    'lstm':        lstm_preds,
    'transformer': trans_preds,
}, index=forecast_dates)

comparison = dl_df.join(garch_df[['garch_forecast']], how='inner')
comparison.rename(columns={'garch_forecast': 'garch'}, inplace=True)

print(f"Aligned: {len(comparison)} rows | {comparison.index[0].date()} → {comparison.index[-1].date()}")
comparison.head(3)

## Results

In [ ]:
def directional_accuracy(actual, pred):
    return (np.sign(np.diff(actual)) == np.sign(np.diff(pred))).mean()

act = comparison['actual'].values

rows = {}
for name in ['garch', 'lstm', 'transformer']:
    p = comparison[name].values
    rows[name] = {
        'MSE':       mean_squared_error(act, p),
        'MAE':       mean_absolute_error(act, p),
        'Dir. Acc.': directional_accuracy(act, p),
    }

metrics_df = pd.DataFrame(rows).T
display_df = metrics_df.copy()
display_df['MSE']       = display_df['MSE'].map('{:.2e}'.format)
display_df['MAE']       = display_df['MAE'].map('{:.5f}'.format)
display_df['Dir. Acc.'] = display_df['Dir. Acc.'].map('{:.1%}'.format)
display_df.index.name   = 'Model'
display_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(comparison.index, comparison['actual'],      label='Actual',      color='black', linewidth=1.5)
ax.plot(comparison.index, comparison['garch'],       label='GARCH(1,1)',  linewidth=1,   alpha=0.8, linestyle='--')
ax.plot(comparison.index, comparison['lstm'],        label='LSTM',        linewidth=1,   alpha=0.8)
ax.plot(comparison.index, comparison['transformer'], label='Transformer', linewidth=1,   alpha=0.8)

ax.set_title('BTC Realized Volatility — 2024 Test Set (All Models)')
ax.set_ylabel('30-day Rolling Volatility')
ax.legend()
plt.tight_layout()
plt.show()

## Conclusion

| Model | MSE | MAE | Dir. Acc. | Characteristic |
|---|---|---|---|---|
| GARCH(1,1) | 4.15e-05 | 0.00540 | **69.5%** | Smooth, upward-biased; mirrors volatility's slow autocorrelation |
| LSTM | **2.56e-06** | **0.00114** | 49.1% | Lowest magnitude error; sequential bias aligns with rolling vol |
| Transformer | 7.59e-06 | 0.00225 | 47.9% | Intermediate error; smoother training but no magnitude win |

### Why GARCH wins on directional accuracy

GARCH forecasts change slowly and track the smooth autocorrelation structure of 30-day rolling volatility. LSTM and Transformer minimize squared error, which pushes them toward near-constant predictions — correctly sized, but stationary in direction.

### Why directional accuracy is the wrong headline metric

The 30-day rolling standard deviation shifts by at most 1/30 of the window per day. Day-over-day direction is dominated by noise, not predictable structure. **MSE and MAE on the 2024 test set are the correct comparison criteria.** On those metrics, LSTM wins by ~16× over GARCH.